# 1. Implementing an RNN for Text Generation

### Task: 
Recurrent Neural Networks (RNNs) can generate sequences of text. You will train an LSTM-based RNN to predict the next character in a given text dataset.

This task implements an LSTM-based recurrent neural network for character-level
text generation. The model will learn patterns in a text dataset and predict
the next character based on a sequence of previous characters.

In [1]:
# Import the required libraries.
# TensorFlow is used to load the dataset and later build the LSTM model.
# NumPy is used for numerical operations during text processing and generation.
import tensorflow as tf
import numpy as np


# 1. Load a text dataset (e.g., "Shakespeare Sonnets", "The Little Prince").

# Download the Shakespeare text dataset using TensorFlow.
# get_file() downloads the file the first time and stores it locally,
# so it does not need to be downloaded again every time the notebook runs.
path_to_file = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

# Open the downloaded text file and read all of its contents.
# UTF-8 is used so that the text characters are read correctly.
with open(path_to_file, "r", encoding="utf-8") as file:
    text = file.read()


# Display basic information about the dataset.
print("Total number of characters:", len(text))

# Display the first 500 characters so we can inspect
# the text before converting it into numerical data.
print("\nFirst 500 characters:\n")
print(text[:500])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total number of characters: 1115394

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [2]:
# 2. Convert text into a sequence of characters (one-hot encoding or embeddings).

# Create a vocabulary containing every unique character in the text.
vocab = sorted(set(text))

# Assign a unique integer ID to each character.
# The LSTM cannot process raw text, so characters must first be represented numerically.
char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = np.array(vocab)

# Convert the entire text from characters to integer IDs.
# These IDs will later be passed through an Embedding layer in the RNN model.
text_as_int = np.array([char_to_index[char] for char in text])

# Display a small sample to confirm that the conversion worked.
print("Number of unique characters:", len(vocab))
print("\nFirst 20 characters:")
print(repr(text[:20]))
print("\nFirst 20 character IDs:")
print(text_as_int[:20])

Number of unique characters: 65

First 20 characters:
'First Citizen:\nBefor'

First 20 character IDs:
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]


In [3]:
# 3. Define an RNN model using LSTM layers to predict the next character.

# Set the size of the character vocabulary.
# The model has one possible output for each unique character.
vocab_size = len(vocab)

# Define the size of the learned character embeddings and the number of memory units in the LSTM layer.
embedding_dim = 256
lstm_units = 512

# Build the LSTM-based RNN model.
model = tf.keras.Sequential([
    
    # Convert each character ID into a learned numerical representation.
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),
    
    # Process the character sequence while remembering information from earlier characters in the sequence.
    tf.keras.layers.LSTM(
        lstm_units,
        return_sequences=True
    ),
    
    # Produce a prediction score for every possible next character.
    tf.keras.layers.Dense(vocab_size)
])

# Build the model with an example input shape so that
# the model architecture can be displayed before training.
model.build(input_shape=(None, None))

# Display the structure and number of parameters in the model.
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 256)      │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 65)       │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,624,897 (6.20 MB)

 Trainable params: 1,624,897 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# 4. Train the model and generate new text by sampling characters one at a time.

# Create sequences of 100 characters and their next-character targets.
sequence_length = 100

dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
dataset = dataset.batch(sequence_length + 1, drop_remainder=True)

# For each sequence, use the first 100 characters as input and the following 100 characters as the expected output.
dataset = dataset.map(lambda x: (x[:-1], x[1:]))

# Shuffle and group the sequences into batches for efficient training.
dataset = dataset.shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)


# Compile and train the LSTM model.
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

history = model.fit(dataset, epochs=5)



# Generate new text by sampling one character at a time.
start_string = "First Citizen:"
input_ids = [char_to_index[char] for char in start_string]
generated_text = start_string

for _ in range(500):
    input_tensor = tf.expand_dims(input_ids[-sequence_length:], 0)

    # Predict scores for the next character.
    predictions = model(input_tensor, training=False)[:, -1, :]

    # Sample one character from the predicted scores.
    predicted_id = tf.random.categorical(predictions, 1)[0, 0].numpy()

    # Add the predicted character to the generated text
    # and use it when predicting the following character.
    generated_text += index_to_char[predicted_id]
    input_ids.append(predicted_id)

print(generated_text)

Epoch 1/5


C:\Users\paran\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


173/173 ━━━━━━━━━━━━━━━━━━━━ 84s 464ms/step - loss: 2.5741 
Epoch 2/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 83s 475ms/step - loss: 1.9465 
Epoch 3/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 82s 470ms/step - loss: 1.7305 
Epoch 4/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 83s 477ms/step - loss: 1.6059 
Epoch 5/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 82s 468ms/step - loss: 1.5269 
First Citizen:
All mathy him; you dreaks both, is not blood much wrait: I'll spair nemers
What to such but out her haslloy:
My best Coriol, lear not Balist, resoration:
He some! my love. exford, God like,
Thun is by grovents thy dendish'd prophosed
master prowe-prigon, and gower oy and would honour
Inde as it a flowarablel-spulp
this fache be bown, for you spild, and make would him,
And sime is my forgain.

FRIARD IIX:
These true deed Tagillan that, to itself.

YORK:
I more the mase on't I prayes him.

GLOUCE


### 5. Temperature Scaling

Temperature scaling controls **how random the model is when selecting the next character** during text generation.

For example, suppose the model predicts that the next character could be:

* `t` = very likely
* `h` = somewhat likely
* `a` = less likely
* `z` = very unlikely

Temperature changes **how strongly the model follows these differences** when sampling the next character.

* **Low temperature (e.g., 0.5):** The model strongly favors characters that already have high prediction scores. For example, it would be much more likely to choose `t` than `a` or `z`. This produces **less random, more predictable text**.

* **Temperature = 1.0:** The model samples directly from its original prediction scores without making the differences stronger or weaker. This is essentially the type of sampling used in Step 4.

* **High temperature (e.g., 1.5):** The differences between the character choices become less dominant, giving lower-scoring characters such as `a` or `z` a greater chance of being selected. This produces **more varied and random text**, but the text may also become less meaningful or coherent.

Therefore:

**Lower temperature - less randomness and more predictable text**

**Higher temperature - more randomness and more varied text**


### Observation

The Shakespeare text dataset contained **1,115,394 characters and 65 unique characters**. The characters were converted into integer IDs so they could be processed by the neural network, and an Embedding layer was used to learn numerical representations of these characters. An LSTM-based RNN was then created to learn the sequence patterns in the text and predict the next character.

During training, the loss decreased consistently from **2.5741 in the first epoch to 1.5269 in the fifth epoch**, showing that the model improved its next-character predictions as it learned from the Shakespeare text. Using **"First Citizen:"** as the starting text, the trained model generated new text by predicting and sampling one character at a time. Although some generated words were not meaningful, the output showed that the model had learned important patterns from the dataset, such as word-like structures, punctuation, line breaks, capitalization, and speaker-style dialogue.

Finally, temperature scaling showed how the randomness of generated text can be controlled. A lower temperature favors more likely characters and produces more predictable text, while a higher temperature gives less likely characters a greater chance of being selected, making the output more varied but potentially less coherent. Overall, the experiment demonstrated how an LSTM can learn character-level patterns from text and use those patterns to generate new sequences.
